# TelecomTS Model Evaluation

Evaluate **Mantis** and **TimesNet** on the [TelecomTS](https://huggingface.co/datasets/AliMaatouk/TelecomTS) dataset for two tasks:

1. **Anomaly Detection** — binary classification (anomaly vs. normal)
2. **Root Cause Analysis** — multi-class classification (10 synthetic anomaly types)

Uses the [TelecomTS benchmark pipeline](https://github.com/Ali-maatouk/TelecomTS)
with the exact encoder architectures and hyperparameters from the paper.

Reference: [APPENG-5739](https://redhat.atlassian.net/browse/APPENG-5739)

## 1. Setup

In [ ]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps", "git+https://github.com/vfeofanov/mantis.git",
])

In [ ]:
!git clone --depth 1 https://github.com/Ali-maatouk/TelecomTS.git /tmp/TelecomTS 2>/dev/null || echo "Already cloned"

import sys
sys.path.insert(0, "/tmp/TelecomTS/src")

In [ ]:
import yaml
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    ConfusionMatrixDisplay,
)

from utils.data_utils import preprocess
from utils.train_utils import prepare, evaluate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2. Load TelecomTS Dataset

32K time series samples from a 5G testbed.  
18 KPI channels (PHY, MAC, Network layers), 128 timesteps each, sampled at 10 Hz.

In [ ]:
dataset = load_dataset(
    "AliMaatouk/TelecomTS",
    data_files={"full": "**/chunked.jsonl"},
)["full"]

splits = dataset.train_test_split(test_size=0.2, seed=SEED)
train_data = list(splits["train"])
test_data = list(splits["test"])

random.shuffle(train_data)

print(f"Train: {len(train_data):,} samples")
print(f"Test:  {len(test_data):,} samples")
print(f"Total: {len(train_data) + len(test_data):,} samples")

In [ ]:
sample = train_data[0]
print(f"Keys: {list(sample.keys())}")
print(f"KPIs: {list(sample['KPIs'].keys())}")
print(f"Anomaly exists: {sample['anomalies']['exists']}")
print(f"Anomaly type: {sample['anomalies'].get('type', 'N/A')}")
print(f"Labels: {sample['labels']}")

## 3. Config

Hyperparameters from the [TelecomTS config](https://github.com/Ali-maatouk/TelecomTS/blob/main/configs/config.yaml).

In [ ]:
with open("/tmp/TelecomTS/configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

TRAIN_CFG = config["train"]
TIMESNET_CFG = config["TimesNet_model"]
MANTIS_CFG = config["Mantis_model"]

print("Training config:", TRAIN_CFG)
print("\nTimesNet config:", TIMESNET_CFG)
print("\nMantis config:", MANTIS_CFG)

## 4. Training + Evaluation Helpers

In [ ]:
from torch.utils.data import DataLoader, TensorDataset


def make_config(encoder_type, task_type, base_config):
    """Build a config dict matching TelecomTS's prepare() expectations."""
    return {
        "encoder_type": encoder_type,
        "task_type": task_type,
        "seed": SEED,
        "train": base_config["train"],
        f"{encoder_type}_model": base_config[f"{encoder_type}_model"],
    }


def train_model(cfg, X_train, y_train, epochs=None):
    """Train an encoder + head using TelecomTS's prepare() utility."""
    model, head, train_dataset, train_dataloader, optimizer, criterion = prepare(
        cfg, X_train, y_train
    )

    model = model.to(DEVICE)
    head = head.to(DEVICE)

    if epochs is None:
        epochs = cfg["train"]["epochs"]

    model.train()
    head.train()

    for epoch in range(epochs):
        losses = []
        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
            optimizer.zero_grad()
            outputs = model(x.permute(0, 2, 1))
            logits = head(outputs)
            loss = criterion(logits, y)
            losses.append(loss.item())
            loss.backward()
            optimizer.step()

        print(f"  Epoch {epoch+1}/{epochs} — loss: {np.mean(losses):.4f}")

    return model, head, train_dataset


def predict(model, head, X, batch_size=64):
    """Get predictions from encoder + head."""
    model.eval()
    head.eval()
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.zeros(len(X), dtype=torch.long),
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            out = head(model(xb.permute(0, 2, 1)))
            preds.append(out.argmax(dim=1).cpu().numpy())

    return np.concatenate(preds)

---
## 5. Task 1 — Anomaly Detection (Binary)

Classify each 128-timestep sample as **normal** (0) or **anomalous** (1).

In [ ]:
X_train_ad, y_train_ad = preprocess(train_data, "anomaly detection")
X_test_ad, y_test_ad = preprocess(test_data, "anomaly detection")

print(f"Train: {X_train_ad.shape}, labels: {np.bincount(y_train_ad)}")
print(f"Test:  {X_test_ad.shape}, labels: {np.bincount(y_test_ad)}")

### 5a. Mantis — Anomaly Detection

In [ ]:
cfg_mantis_ad = make_config("Mantis", "anomaly detection", config)

print("Training Mantis for anomaly detection...")
mantis_ad_model, mantis_ad_head, _ = train_model(cfg_mantis_ad, X_train_ad, y_train_ad)

y_pred_mantis_ad = predict(mantis_ad_model, mantis_ad_head, X_test_ad)

print("\n=== Mantis — Anomaly Detection ===")
print(classification_report(y_test_ad, y_pred_mantis_ad, target_names=["Normal", "Anomaly"]))

### 5b. TimesNet — Anomaly Detection

In [ ]:
cfg_timesnet_ad = make_config("TimesNet", "anomaly detection", config)

print("Training TimesNet for anomaly detection...")
timesnet_ad_model, timesnet_ad_head, _ = train_model(cfg_timesnet_ad, X_train_ad, y_train_ad)

y_pred_timesnet_ad = predict(timesnet_ad_model, timesnet_ad_head, X_test_ad)

print("\n=== TimesNet — Anomaly Detection ===")
print(classification_report(y_test_ad, y_pred_timesnet_ad, target_names=["Normal", "Anomaly"]))

---
## 6. Task 2 — Root Cause Analysis (Multi-class)

For anomalous samples only: classify the cause among 10 synthetic anomaly types.  
(Jamming is excluded — it's the single real anomaly and treated separately.)

In [ ]:
RCA_CLASSES = [
    "Antenna Failure",
    "Co-Channel Interference (Mild)",
    "Co-Channel Interference (Severe)",
    "Faulty RF Filters (Temporal)",
    "Doppler Shift (Severe)",
    "Faulty Handover Algorithm (Too Frequent)",
    "Buffer Overflow (Gradual Buildup)",
    "Resource Allocation Bugs",
    "High Network Congestion (Gradual Buildup)",
    "High Network Congestion (Sudden Spike)",
]

X_train_rca, y_train_rca = preprocess(train_data, "root-cause analysis")
X_test_rca, y_test_rca = preprocess(test_data, "root-cause analysis")

print(f"Train: {X_train_rca.shape}, class distribution: {np.bincount(y_train_rca)}")
print(f"Test:  {X_test_rca.shape}, class distribution: {np.bincount(y_test_rca)}")

### 6a. Mantis — Root Cause Analysis

In [ ]:
cfg_mantis_rca = make_config("Mantis", "root-cause analysis", config)

print("Training Mantis for root cause analysis...")
mantis_rca_model, mantis_rca_head, _ = train_model(cfg_mantis_rca, X_train_rca, y_train_rca)

y_pred_mantis_rca = predict(mantis_rca_model, mantis_rca_head, X_test_rca)

print("\n=== Mantis — Root Cause Analysis ===")
print(classification_report(y_test_rca, y_pred_mantis_rca, target_names=RCA_CLASSES))

### 6b. TimesNet — Root Cause Analysis

In [ ]:
cfg_timesnet_rca = make_config("TimesNet", "root-cause analysis", config)

print("Training TimesNet for root cause analysis...")
timesnet_rca_model, timesnet_rca_head, _ = train_model(cfg_timesnet_rca, X_train_rca, y_train_rca)

y_pred_timesnet_rca = predict(timesnet_rca_model, timesnet_rca_head, X_test_rca)

print("\n=== TimesNet — Root Cause Analysis ===")
print(classification_report(y_test_rca, y_pred_timesnet_rca, target_names=RCA_CLASSES))

---
## 7. Results Comparison

In [ ]:
results = {}
for task, y_true, preds in [
    ("Anomaly Detection", y_test_ad, [("Mantis", y_pred_mantis_ad), ("TimesNet", y_pred_timesnet_ad)]),
    ("Root Cause Analysis", y_test_rca, [("Mantis", y_pred_mantis_rca), ("TimesNet", y_pred_timesnet_rca)]),
]:
    results[task] = {}
    for model_name, y_pred in preds:
        results[task][model_name] = {
            "Accuracy": accuracy_score(y_true, y_pred),
            "F1 (macro)": f1_score(y_true, y_pred, average='macro', zero_division=0),
            "F1 (weighted)": f1_score(y_true, y_pred, average='weighted', zero_division=0),
        }

for task in results:
    print(f'\n=== {task} ===')
    for model_name, metrics in results[task].items():
        print(f'  {model_name}: ' + ', '.join(f'{k}={v:.3f}' for k, v in metrics.items()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Anomaly Detection
ad_metrics = list(results["Anomaly Detection"]["Mantis"].keys())
mantis_ad_vals = [results["Anomaly Detection"]["Mantis"][m] for m in ad_metrics]
timesnet_ad_vals = [results["Anomaly Detection"]["TimesNet"][m] for m in ad_metrics]

x = np.arange(len(ad_metrics))
width = 0.3
axes[0].bar(x - width / 2, mantis_ad_vals, width, label="Mantis", color="#2196F3")
axes[0].bar(x + width / 2, timesnet_ad_vals, width, label="TimesNet", color="#FF9800")
axes[0].set_xticks(x)
axes[0].set_xticklabels(ad_metrics)
axes[0].set_ylabel("Score")
axes[0].set_title("Anomaly Detection")
axes[0].legend()
axes[0].set_ylim(0, 1)

# Root Cause Analysis
rca_metrics = list(results["Root Cause Analysis"]["Mantis"].keys())
mantis_rca_vals = [results["Root Cause Analysis"]["Mantis"][m] for m in rca_metrics]
timesnet_rca_vals = [results["Root Cause Analysis"]["TimesNet"][m] for m in rca_metrics]

x = np.arange(len(rca_metrics))
axes[1].bar(x - width / 2, mantis_rca_vals, width, label="Mantis", color="#2196F3")
axes[1].bar(x + width / 2, timesnet_rca_vals, width, label="TimesNet", color="#FF9800")
axes[1].set_xticks(x)
axes[1].set_xticklabels(rca_metrics)
axes[1].set_ylabel("Score")
axes[1].set_title("Root Cause Analysis")
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.suptitle("Mantis vs TimesNet — TelecomTS", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

for ax, (y_true, y_pred, title) in zip(
    axes.flat,
    [
        (y_test_ad, y_pred_mantis_ad, "Mantis — Anomaly Detection"),
        (y_test_ad, y_pred_timesnet_ad, "TimesNet — Anomaly Detection"),
        (y_test_rca, y_pred_mantis_rca, "Mantis — Root Cause Analysis"),
        (y_test_rca, y_pred_timesnet_rca, "TimesNet — Root Cause Analysis"),
    ],
):
    if "Root Cause" in title:
        labels = [c[:20] for c in RCA_CLASSES]
    else:
        labels = ["Normal", "Anomaly"]

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(title, fontsize=12)
    if "Root Cause" in title:
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        ax.set_yticklabels(labels, fontsize=8)

plt.suptitle("Confusion Matrices", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()